# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irssaa29/Machine-learning_01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (revised): A page is worth reviewing if it ranks well enough to have real position data (not no_data), but underperforms the CTR expected for its position tier, and has enough impression volume (>=100) to make fixing it worthwhile. Declining status (is_declining) is deliberately NOT used as an input to this rule — only observed, pre-decision signals from March 1-15 are used, to avoid any risk of leaking the label into the score.

Signal 1 (flag-linked — CTR-fix logic): hypothesis — pages with worse average position have lower CTR. Checked by bucketing pages into position tiers and comparing mean CTR per bucket.

Signal 2 (flag-linked — volume/quick-win logic): hypothesis — impression volume is roughly independent of decline risk, meaning volume is a genuine multiplier on priority (bigger opportunity when it does decline) rather than a duplicate signal of decline itself. Checked by bucketing pages into volume tiers and comparing decline rate per bucket.

Reason codes this rule can output: position_ctr_gap (poor CTR relative to position tier).

Note: high_volume_decline was originally planned as a second reason code, but was dropped after Signal 2's verdict below showed volume is not independent of decline — using it risked near-circularity with the label. Only position_ctr_gap is used in the final rule.

In [4]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/irssaa29/Machine-learning_01"
REPO_DIR = "Machine-learning_01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

# --- Reload March data and rebuild the same df_pair from w03 ---
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/"
)
df_march["report_date"] = pd.to_datetime(df_march["report_date"])
first_half = df_march[df_march["report_date"].dt.day <= 15]
second_half = df_march[df_march["report_date"].dt.day > 15]

feat = first_half.groupby("content_hash_id").agg(
    h1_impressions=("gsc_impressions", "sum"),
    h1_avg_position=("gsc_avg_position", "mean"),
    h1_clicks=("gsc_clicks", "sum"),
    h1_active_days=("report_date", "nunique"),
).reset_index()
feat["h1_ctr"] = feat["h1_clicks"] / feat["h1_impressions"].replace(0, pd.NA)

h2 = second_half.groupby("content_hash_id").agg(h2_clicks=("gsc_clicks", "sum")).reset_index()
df_pair = feat.merge(h2, on="content_hash_id", how="inner")
df_pair["is_declining"] = (df_pair["h2_clicks"] < df_pair["h1_clicks"]).astype(int)

print("Loaded:", df_pair.shape)

# --- Signal 1: position tiers vs CTR ---
def position_tier(p):
    if pd.isna(p) or p == 0:
        return "no_data"
    elif p <= 10:
        return "good_1_10"
    elif p <= 30:
        return "mid_11_30"
    else:
        return "poor_30plus"

df_pair["position_tier"] = df_pair["h1_avg_position"].apply(position_tier)

signal1_table = df_pair.groupby("position_tier").agg(
    n=("content_hash_id", "count"),
    mean_ctr=("h1_ctr", "mean"),
).reset_index()
print("\n--- Signal 1: Position tier vs CTR ---")
print(signal1_table)
print(df_pair[df_pair["position_tier"] == "no_data"]["h1_impressions"].describe())
print(df_pair[df_pair["position_tier"] == "no_data"]["h1_clicks"].describe())

# --- Signal 2: impression volume tiers vs decline rate ---
zero_mask = df_pair["h1_impressions"] == 0

df_pair["volume_tier"] = "zero"
nonzero = df_pair.loc[~zero_mask, "h1_impressions"]

tier_labels = pd.qcut(nonzero, q=4, labels=["low", "mid_low", "mid_high", "high"], duplicates="drop")
df_pair.loc[~zero_mask, "volume_tier"] = tier_labels.astype(str)

signal2_table = df_pair.groupby("volume_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    decline_rate=("is_declining", "mean"),
).reset_index()
print("\n--- Signal 2: Volume tier vs Decline rate ---")
print(signal2_table)

Loaded: (319758, 8)

--- Signal 1: Position tier vs CTR ---
  position_tier       n  mean_ctr
0     good_1_10   86251  0.005533
1     mid_11_30   40763  0.003177
2       no_data  169084  0.031649
3   poor_30plus   23660  0.001645
count    169084.000000
mean          0.016223
std           0.418167
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          76.000000
Name: h1_impressions, dtype: float64
count    169084.000000
mean          0.000272
std           0.016847
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           2.000000
Name: h1_clicks, dtype: float64

--- Signal 2: Volume tier vs Decline rate ---
  volume_tier       n  decline_rate
0        high   37965      0.429870
1         low   38711      0.026142
2    mid_high   37886      0.229161
3     mid_low   37418      0.079507
4        zero  167778      0.000000


Signal 1 verdict: *CONFIRMED.* Among pages with real position data, CTR decreases monotonically as position worsens (good_1_10: 0.0055 → mid_11_30: 0.0032 → poor_30plus: 0.0016). The no_data tier's inflated mean CTR (0.0316) is a division artifact — median impressions and clicks are both 0 for this group, so a handful of tiny-denominator pages (e.g. 1 click / 1 impression) skew the average. These rows are excluded from the rule.

Signal 2 verdict: *OPPOSITE.* Hypothesized volume would be roughly independent of decline risk; instead decline rate rises steeply with volume (zero: 0.0% → low: 2.6% → mid_low: 8.0% → mid_high: 22.9% → high: 43.0%). Part of this is structural: a page with zero first-half clicks cannot mathematically decline under this label definition. This means raw volume can't be used as an independent multiplier without risking near-circularity with the label — so in the rule, volume is used only as a floor/filter, not a multiplier on severity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# --- Filter out no_data noise pages first ---
df_valid = df_pair[df_pair["position_tier"] != "no_data"].copy()
print("Valid pages (real position data):", len(df_valid))

# --- Expected CTR benchmark per position tier (from Signal 1 table) ---
tier_benchmark = df_valid.groupby("position_tier")["h1_ctr"].transform("mean")
df_valid["ctr_gap"] = tier_benchmark - df_valid["h1_ctr"]  # positive = underperforming for its tier

# --- Rule: flag pages with a real CTR gap AND meaningful volume (floor, not multiplier) ---
min_impressions_floor = 100  # meaningful traffic floor
df_valid["score"] = np.where(
    df_valid["h1_impressions"] >= min_impressions_floor,
    df_valid["ctr_gap"].clip(lower=0) * df_valid["h1_impressions"],
    0
)

df_valid["reason_code"] = np.where(
    df_valid["score"] > 0, "position_ctr_gap", "none"
)
df_valid["action"] = np.where(
    df_valid["score"] > 0, "review_for_ctr_fix", "monitor"
)

df_ranked = df_valid.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
df_ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(df_ranked[["content_hash_id", "position_tier", "h1_ctr", "h1_impressions", "score", "reason_code", "action"]].head(10))


Valid pages (real position data): 150674
            content_hash_id position_tier    h1_ctr  h1_impressions  \
0  content_9c057b66c30a3abb     good_1_10       0.0           83772   
1  content_7c6373141eae744a     good_1_10  0.000587           86860   
2  content_34a70fea29d15f24     good_1_10  0.000244           73639   
3  content_acbcc847f8996314     good_1_10  0.001589           83715   
4  content_8e1334d6356668e3     good_1_10  0.000017           58553   
5  content_b99ea6861864dea5     good_1_10  0.002022           91474   
6  content_65c75874a23fca87     good_1_10  0.000269           55680   
7  content_1642f339bd6e7c8d     good_1_10  0.000344           52378   
8  content_945d6ff91386c817     good_1_10  0.000041           49314   
9  content_f6116743b00afc2d     good_1_10  0.000161           49619   

        score       reason_code              action  
0  463.492969  position_ctr_gap  review_for_ctr_fix  
1  429.578227  position_ctr_gap  review_for_ctr_fix  
2  389.429197  

In [8]:
import json

metrics = {
    "run_date": "2026-08-13",
    "lane": "Refresh / Content Opportunity Scoring",
    "month_analyzed": "2026-03",
    "total_pages_loaded": len(df_pair),
    "valid_pages_scored": len(df_valid),
    "pages_flagged_for_review": int((df_valid["score"] > 0).sum()),
    "signal_1_verdict": "CONFIRMED",
    "signal_2_verdict": "OPPOSITE",
    "top_score": float(df_ranked["score"].max()),
    "reason_codes_used": ["position_ctr_gap"],
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics:")
print(json.dumps(metrics, indent=2))

Saved metrics:
{
  "run_date": "2026-08-13",
  "lane": "Refresh / Content Opportunity Scoring",
  "month_analyzed": "2026-03",
  "total_pages_loaded": 319758,
  "valid_pages_scored": 150674,
  "pages_flagged_for_review": 59761,
  "signal_1_verdict": "CONFIRMED",
  "signal_2_verdict": "OPPOSITE",
  "top_score": 463.4929688215155,
  "reason_codes_used": [
    "position_ctr_gap"
  ]
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top10 = df_ranked.head(10)[["content_hash_id", "position_tier", "h1_ctr", "h1_impressions", "score", "reason_code", "action"]]
print(top10.to_string(index=False))


         content_hash_id position_tier    h1_ctr  h1_impressions       score      reason_code             action
content_9c057b66c30a3abb     good_1_10       0.0           83772  463.492969 position_ctr_gap review_for_ctr_fix
content_7c6373141eae744a     good_1_10  0.000587           86860  429.578227 position_ctr_gap review_for_ctr_fix
content_34a70fea29d15f24     good_1_10  0.000244           73639  389.429197 position_ctr_gap review_for_ctr_fix
content_acbcc847f8996314     good_1_10  0.001589           83715    330.1776 position_ctr_gap review_for_ctr_fix
content_8e1334d6356668e3     good_1_10  0.000017           58553  322.961512 position_ctr_gap review_for_ctr_fix
content_b99ea6861864dea5     good_1_10  0.002022           91474  321.106525 position_ctr_gap review_for_ctr_fix
content_65c75874a23fca87     good_1_10  0.000269           55680  293.065804 position_ctr_gap review_for_ctr_fix
content_1642f339bd6e7c8d     good_1_10  0.000344           52378  271.796528 position_ctr_gap re

1. *content_9c057b66c30a3abb* — action: review_for_ctr_fix. Ranking well (top 10 position tier) with 83,772 impressions but CTR near 0 — the single largest volume-weighted CTR gap. Would be wrong if the listing is intentionally a low-click reference page (e.g., an internal doc) rather than a page meant to convert clicks.
2. *content_7c6373141eae744a* — action: review_for_ctr_fix. 86,860 impressions, CTR 0.0006 — massive visibility with almost no clicks. Would be wrong if this page duplicates a better-performing page and traffic is intentionally routed elsewhere.
3. *content_34a70fea29d15f24* — action: review_for_ctr_fix. 73,639 impressions, CTR 0.0002 — among the lowest CTRs in the whole top 10. Would be wrong if the low CTR reflects a seasonal dip rather than a genuine title/snippet problem.
4. *content_acbcc847f8996314* — action: review_for_ctr_fix. 83,715 impressions, CTR 0.0016 — slightly higher CTR than others here but still far below the tier average. Would be wrong if this page recently changed and hasn't accumulated enough post-change data yet.
5. *content_8e1334d6356668e3* — action: review_for_ctr_fix. 58,553 impressions, CTR 0.00002 — essentially zero clicks despite strong visibility. Would be wrong if this page is non-clickable by design (e.g., an image result or rich snippet answer that satisfies the query without a click).
6. *content_b99ea6861864dea5* — action: review_for_ctr_fix. 91,474 impressions (highest in top 10), CTR 0.002 — largest audience in the list. Would be wrong if the query intent here is informational and a low click rate is actually expected/normal for that query type.
7. *content_65c75874a23fca87* — action: review_for_ctr_fix. 55,680 impressions,TR 0.0003. Would be wrong if this page's true ranking position fluctuates heavily day-to-day and the March average masks a recent genuine improvement.
8. *content_1642f339bd6e7c8d* — action: review_for_ctr_fix. 52,378 impressions, CTR 0.0003. Would be wrong if impressions are inflated by broad/irrelevant query matches that were never going to convert to clicks regardless of snippet quality.
9. *content_945d6ff91386c817* — action: review_for_ctr_fix. 49,314 impressions, CTR 0.00004 — near-zero CTR. Would be wrong if this page is cannibalizing clicks from a sibling page on the same site that ranks even higher for the same query.
10. *content_f6116743b00afc2d* — action: review_for_ctr_fix. 49,619 impressions, CTR 0.0002. Would be wrong if this content_hash_id represents a redirect or deprecated URL still being indexed, where a CTR fix wouldn't actually be actionable.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

*Weak pick*: all ten top picks are structurally identical (same tier, same reason code, same failure story), which is itself a weakness — my rule isn't distinguishing between different opportunity types, just ranking by raw impression-weighted CTR gap. A stronger rule would surface some diversity — e.g., a "poor_30plus tier, high volume" pick — instead of one repeated pattern. This suggests my score formula over-weights impression volume relative to CTR gap severity, since score = ctr_gap * impressions lets sheer traffic size dominate the ranking.

*Leakage check*: confirmed no future-window or label-derived inputs were used in the rule. h1_ctr, h1_avg_position, and h1_impressions are all computed strictly from March 1–15 (the "before" window), and is_declining/h2_clicks (the March 16–31 "after" data) were never referenced anywhere in the scoring formula. No product-decision flags exist in this warehouse table to begin with.

In [7]:
scoring_columns_used = ["h1_ctr", "h1_avg_position", "h1_impressions", "position_tier", "ctr_gap"]
label_derived_columns = ["is_declining", "h2_clicks"]

leaked = [c for c in scoring_columns_used if c in label_derived_columns]
print("Leaked columns found in scoring inputs:", leaked if leaked else "NONE — confirmed clean")

Leaked columns found in scoring inputs: NONE — confirmed clean


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.